# 0.7 · 线性代数 / Linear Algebra

> **课程定位 / Where this fits**
> 第 7 课，**Part 0 · 基础准备**。
> Lesson 7, **Part 0 · Foundations**.
>
> **数据科学的所有"硬核"算法——PCA、推荐系统的矩阵分解、神经网络的反向传播、SVM 的核技巧——都站在线性代数上面**。本节我们用代数 + 几何 + NumPy 三视角把它一次性打通。
> **Every "hard-core" DS algorithm stands on linear algebra** — PCA, matrix factorization in recommenders, NN backprop, SVM kernels. We'll cover it from three angles: algebra + geometry + NumPy.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $x, y, c$ —— 普通小写斜体 = 标量 / scalar
> - $\mathbf{x}, \mathbf{v}, \mathbf{w}$ —— **粗体小写 = 列向量**（默认）/ bold lowercase = column vector
> - $\mathbf{A}, \mathbf{B}, \mathbf{X}$ —— **粗体大写 = 矩阵** / bold uppercase = matrix
> - $\mathbf{X}^\top$ —— 转置 / transpose
> - $\langle\mathbf{x}, \mathbf{y}\rangle = \mathbf{x}^\top \mathbf{y}$ —— 内积 / inner product
> - $\|\mathbf{x}\| \equiv \|\mathbf{x}\|_2$ —— 默认 L2 范数 / default L2 norm
> - $\mathbb{R}^d$ —— $d$ 维实向量空间 / $d$-dim real vector space

> 💡 **面试相关 / Interview-relevant**
> 线性代数在 DS 面试里经常以"**手推**"的方式出现：让你白板上推 PCA、SVD、最小二乘的正规方程。本节所有定理我都给出推导，**白板时不慌**。
> Linear algebra often shows up as whiteboard derivations in DS interviews (PCA / SVD / normal equation). All theorems here include derivations so you won't freeze at the whiteboard.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 把向量看成"几何箭头"和"代数数组"两种**等价视角**自由切换。
   Switch fluently between **geometric (arrows)** and **algebraic (arrays)** views of vectors.
2. 理解 $\mathbf{A}\mathbf{x}$ 既是"代数乘法"又是"**对 $\mathbf{x}$ 做线性变换**"。
   Understand $\mathbf{A}\mathbf{x}$ both as a product and as a **linear transformation** of $\mathbf{x}$.
3. 自己证明：对称矩阵 → 实特征值 + 正交特征向量。
   Prove: symmetric matrix ⇒ real eigenvalues + orthogonal eigenvectors.
4. 解释 SVD $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$ 几何上"**旋转-拉伸-旋转**"的含义。
   Explain SVD geometrically as **rotate–stretch–rotate**.
5. 用 NumPy 手写 PCA，并和 sklearn `PCA` 对比。
   Implement PCA from scratch with NumPy, cross-check against sklearn.

---

## 目录 / Table of Contents

1. [向量 / Vectors](#1)
2. [向量运算 / Vector Operations](#2)
3. [内积、范数、距离、夹角 / Inner Product, Norm, Distance, Angle](#3)
4. [矩阵基础 / Matrix Basics](#4)
5. [**矩阵 = 线性变换** ⭐ / Matrices as Linear Transformations](#5)
6. [矩阵乘法的三种视角 / Three Views of Matmul](#6)
7. [特殊矩阵 / Special Matrices](#7)
8. [行列式 / Determinant —— 几何 = 面积/体积缩放](#8)
9. [秩、列空间、零空间 / Rank, Column Space, Null Space](#9)
10. [矩阵的逆 / Matrix Inverse](#10)
11. [解线性方程组 / Solving Linear Systems](#11)
12. [投影 / Projection](#12)
13. [**特征值与特征向量** ⭐ / Eigenvalues & Eigenvectors](#13)
14. [对称矩阵的谱分解 / Spectral Theorem](#14)
15. [**SVD** ⭐ / Singular Value Decomposition](#15)
16. [应用：在 Iris 上手写 PCA / Application: PCA from Scratch on Iris](#16)
17. [小结 / Summary](#17)


<a id="1"></a>
## 1. 向量 / Vectors

### 1.1 两种等价视角 / Two equivalent views

| 视角 / View | 写法 / Notation | 直觉 / Intuition |
|---|---|---|
| **代数** / Algebraic | $\mathbf{v} = \begin{pmatrix} v_1 \\\\ v_2 \\\\ \vdots \\\\ v_d \end{pmatrix} \in \mathbb{R}^d$ | $d$ 个数排成一列 |
| **几何** / Geometric | 从原点出发的"箭头" | 在 $\mathbb{R}^d$ 中的方向 + 长度 |

> 在数据科学里，一个**样本**就是一个向量：$\mathbf{x}_i \in \mathbb{R}^d$ 表示第 $i$ 个样本的 $d$ 个特征。
> A **sample** is a vector: $\mathbf{x}_i \in \mathbb{R}^d$ = $d$ features of sample $i$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

# 列向量约定 / Column-vector convention
v = np.array([3, 2])           # 2D 向量 / 2D vector
w = np.array([1, 4])

print("v =", v)
print("w =", w)
print(f"v ∈ ℝ^{v.shape[0]}")


In [ ]:
# 把 v 和 w 画成箭头 / Visualize as arrows
def plot_vectors(vecs, labels, colors, ax=None, lim=6):
    # 画一组从原点出发的 2D 向量箭头 / Draw 2D vectors from origin
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    for vec, lab, col in zip(vecs, labels, colors):
        ax.quiver(0, 0, vec[0], vec[1], angles="xy", scale_units="xy",
                  scale=1, color=col, width=0.012)
        ax.text(vec[0]*1.07, vec[1]*1.07, lab, color=col, fontsize=13)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    return ax

plot_vectors([v, w], ["v", "w"], ["red", "blue"])
plt.title("Two vectors in R²")
plt.show()


<a id="2"></a>
## 2. 向量运算 / Vector Operations

### 2.1 加法 / Addition

代数：
$$\mathbf{v} + \mathbf{w} = \begin{pmatrix} v_1 + w_1 \\\\ \vdots \\\\ v_d + w_d \end{pmatrix}$$

几何：**首尾相接**（平行四边形法则）。
Geometric: place $\mathbf{w}$ tip-to-tail at the end of $\mathbf{v}$ (parallelogram rule).

### 2.2 标量乘法 / Scalar multiplication

代数：$c\mathbf{v} = (cv_1, \dots, cv_d)^\top$
几何：**拉伸/反向**。$c > 0$ 同向拉伸；$c < 0$ 反向；$|c| > 1$ 拉长，$|c| < 1$ 压短。
Geometric: stretch (or flip if negative).


In [ ]:
# 加法 + 标量乘法可视化 / Addition + scaling viz
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: v + w
ax = axes[0]
plot_vectors([v, w, v + w], ["v", "w", "v+w"], ["red", "blue", "purple"], ax=ax)
# 显示 w 平移到 v 末端 / show w shifted to tip of v
ax.quiver(v[0], v[1], w[0], w[1], angles="xy", scale_units="xy", scale=1,
          color="blue", alpha=0.4, width=0.008)
ax.set_title("Addition: v + w (head-to-tail)")

# Right: c * v
ax = axes[1]
plot_vectors([v, 2*v, -1*v, 0.5*v],
             ["v", "2v", "-v", "0.5v"],
             ["red", "darkred", "orange", "salmon"], ax=ax)
ax.set_title("Scalar multiplication")

plt.tight_layout()
plt.show()


<a id="3"></a>
## 3. 内积、范数、距离、夹角 / Inner Product, Norm, Distance, Angle

### 3.1 内积 / Inner product

定义 / Definition：

$$
\boxed{\;\langle \mathbf{v}, \mathbf{w} \rangle \;\equiv\; \mathbf{v}^\top \mathbf{w} \;=\; \sum_{i=1}^d v_i w_i\;}
$$

### 3.2 范数（长度）/ Norm (length)

$$\|\mathbf{v}\| = \sqrt{\langle \mathbf{v}, \mathbf{v} \rangle} = \sqrt{v_1^2 + \dots + v_d^2}$$

### 3.3 距离 / Distance

$$d(\mathbf{v}, \mathbf{w}) = \|\mathbf{v} - \mathbf{w}\|$$

### 3.4 夹角 / Angle ⭐

$$
\boxed{\;\cos \theta \;=\; \dfrac{\langle \mathbf{v}, \mathbf{w}\rangle}{\|\mathbf{v}\| \cdot \|\mathbf{w}\|}\;}
$$

由此推出**正交性**：$\mathbf{v} \perp \mathbf{w} \;\iff\; \langle \mathbf{v}, \mathbf{w}\rangle = 0$
Hence **orthogonality**: $\mathbf{v} \perp \mathbf{w}$ iff inner product is 0.

> 💡 **面试常考 / Common interview Q**：余弦相似度（NLP、推荐系统的核心）就是 $\cos\theta$。
> Cosine similarity (NLP, recommenders) **is** just $\cos\theta$.


In [ ]:
# 算一下 v, w 的内积、长度、距离、夹角 / Compute these
v = np.array([3, 2])
w = np.array([1, 4])

inner = v @ w                                # 内积 / inner product
norm_v = np.linalg.norm(v)
norm_w = np.linalg.norm(w)
dist = np.linalg.norm(v - w)
cos_th = inner / (norm_v * norm_w)
theta = np.arccos(cos_th)                    # 弧度 / radians

print(f"⟨v, w⟩  = {inner}")
print(f"‖v‖    = {norm_v:.4f}")
print(f"‖w‖    = {norm_w:.4f}")
print(f"d(v,w) = {dist:.4f}")
print(f"cos θ  = {cos_th:.4f}")
print(f"θ      = {np.degrees(theta):.2f}°")


In [ ]:
# 正交向量演示 / Orthogonal vectors demo
a = np.array([2, 1])
b = np.array([-1, 2])     # 注意 a·b = -2 + 2 = 0 → 正交 / orthogonal

print(f"a · b = {a @ b}")
print(f"orthogonal? {a @ b == 0}")

plot_vectors([a, b], ["a", "b"], ["green", "purple"])
plt.title("a · b = 0  →  orthogonal")
plt.show()


### 3.5 几个重要不等式 / Key inequalities

| 名称 / Name | 公式 / Formula |
|---|---|
| Cauchy–Schwarz | $\lvert\langle \mathbf{v}, \mathbf{w}\rangle\rvert \le \|\mathbf{v}\|\,\|\mathbf{w}\|$ |
| 三角不等式 / Triangle | $\|\mathbf{v} + \mathbf{w}\| \le \|\mathbf{v}\| + \|\mathbf{w}\|$ |

Cauchy–Schwarz 的等号成立当且仅当 $\mathbf{v}, \mathbf{w}$ **共线**。
Equality holds iff $\mathbf{v}, \mathbf{w}$ are **collinear**.


<a id="4"></a>
## 4. 矩阵基础 / Matrix Basics

$$\mathbf{A} \in \mathbb{R}^{m \times n} = \begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1n} \\\\
a_{21} & a_{22} & \cdots & a_{2n} \\\\
\vdots & \vdots & \ddots & \vdots \\\\
a_{m1} & a_{m2} & \cdots & a_{mn}
\end{pmatrix}$$

**形状记忆 / Shape mnemonic**：$\mathbf{A}_{m \times n}$ = "$m$ 行 $n$ 列" / "$m$ rows, $n$ cols"。

数据科学里**设计矩阵** / Design matrix：$\mathbf{X} \in \mathbb{R}^{n \times d}$（$n$ 个样本，$d$ 个特征）。

### 基本属性 / Basic attributes


In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])
print(f"A =\n{A}")
print(f"shape : {A.shape}")        # (m, n)
print(f"A^T   :\n{A.T}")            # 转置 / transpose

# 加法、标量乘法都是 element-wise / Addition and scaling are elementwise
B = np.array([[1, 0, 1], [0, 1, 0]])
print(f"\nA + B =\n{A + B}")
print(f"\n3 * A =\n{3 * A}")


<a id="5"></a>
## 5. 矩阵 = 线性变换 ⭐ / Matrices as Linear Transformations

**这是整章最重要的概念。**
**This is the most important idea of the whole chapter.**

任何 $\mathbf{A} \in \mathbb{R}^{m \times n}$ 都对应一个**线性变换** $T: \mathbb{R}^n \to \mathbb{R}^m$：
Every matrix corresponds to a **linear transformation**:

$$T(\mathbf{x}) = \mathbf{A}\mathbf{x}$$

满足两条性质 / Two properties:

1. **可加性 / additivity**: $T(\mathbf{x} + \mathbf{y}) = T(\mathbf{x}) + T(\mathbf{y})$
2. **齐次性 / homogeneity**: $T(c\mathbf{x}) = c\,T(\mathbf{x})$

合起来叫**线性 / linearity**。


### 5.1 在 $\mathbb{R}^2$ 里"看见"线性变换 / Visualizing in $\mathbb{R}^2$

对一个 $2 \times 2$ 矩阵 $\mathbf{A}$，$T(\mathbf{x}) = \mathbf{A}\mathbf{x}$ 把整个平面"重塑"了。最有名的几个：
A $2 \times 2$ matrix reshapes the whole plane. Famous examples:

| 变换 / Transform | 矩阵 / Matrix |
|---|---|
| 恒等 / Identity | $\begin{pmatrix} 1 & 0 \\\\ 0 & 1 \end{pmatrix}$ |
| 沿 $x$ 拉伸 2 倍 / Stretch x by 2 | $\begin{pmatrix} 2 & 0 \\\\ 0 & 1 \end{pmatrix}$ |
| 旋转 $\theta$ / Rotate by $\theta$ | $\begin{pmatrix} \cos\theta & -\sin\theta \\\\ \sin\theta & \cos\theta \end{pmatrix}$ |
| 沿 $x$ 反射 / Reflect over x-axis | $\begin{pmatrix} 1 & 0 \\\\ 0 & -1 \end{pmatrix}$ |
| 剪切 / Shear | $\begin{pmatrix} 1 & 1 \\\\ 0 & 1 \end{pmatrix}$ |


In [ ]:
# 把一个"单位正方形"用不同矩阵变换 —— 直观感受 / Transform the unit square
square = np.array([[0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0]])      # 单位正方形的 5 个顶点（闭合）

transforms = {
    "Identity":     np.eye(2),
    "Stretch x×2":  np.array([[2, 0], [0, 1]]),
    "Rotate 45°":   np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                              [np.sin(np.pi/4),  np.cos(np.pi/4)]]),
    "Shear":        np.array([[1, 1], [0, 1]]),
    "Reflect over x": np.array([[1, 0], [0, -1]]),
    "Scale×0.5+Rot":  0.5 * np.array([[np.cos(np.pi/3), -np.sin(np.pi/3)],
                                       [np.sin(np.pi/3),  np.cos(np.pi/3)]]),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, A) in zip(axes.flat, transforms.items()):
    out = A @ square
    ax.plot(square[0], square[1], "k--", alpha=0.4, label="original")
    ax.fill(out[0], out[1], alpha=0.3, color="steelblue")
    ax.plot(out[0], out[1], "b-", label="transformed")

    # 显示基向量怎么变 / Show how basis vectors transform
    ax.quiver(0, 0, A[0, 0], A[1, 0], angles="xy", scale_units="xy",
              scale=1, color="red", width=0.012, label="A·e₁")
    ax.quiver(0, 0, A[0, 1], A[1, 1], angles="xy", scale_units="xy",
              scale=1, color="green", width=0.012, label="A·e₂")

    ax.set_xlim(-2, 3); ax.set_ylim(-2, 3)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(name)
    ax.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()


**关键洞察 / Key insight**：
- 红色 / 绿色箭头 = $\mathbf{A}$ 的**两列**就是 $\mathbf{A} \mathbf{e}_1, \mathbf{A} \mathbf{e}_2$（基向量变到哪里）。
- 知道了基向量去哪里，整个变换就完全确定了——线性的力量。
- Red / green arrows = the **columns** of $\mathbf{A}$ = where the basis vectors land. Once you know where basis vectors go, the whole transform is determined.

写成公式 / In formula:
$$\mathbf{A}\mathbf{x} = x_1 \mathbf{A}\mathbf{e}_1 + x_2 \mathbf{A}\mathbf{e}_2 + \dots + x_n \mathbf{A}\mathbf{e}_n$$

即：**$\mathbf{A}\mathbf{x}$ 是 $\mathbf{A}$ 各列的线性组合，组合系数就是 $\mathbf{x}$ 的分量。**
**$\mathbf{A}\mathbf{x}$ is a linear combination of $\mathbf{A}$'s columns with coefficients from $\mathbf{x}$.**


<a id="6"></a>
## 6. 矩阵乘法的三种视角 / Three Views of Matrix Multiplication

设 $\mathbf{A} \in \mathbb{R}^{m\times k}$，$\mathbf{B} \in \mathbb{R}^{k\times n}$，$\mathbf{C} = \mathbf{A}\mathbf{B} \in \mathbb{R}^{m\times n}$。

### 视角 1：标准定义 / Standard definition

$$C_{ij} = \sum_{p=1}^k A_{ip} B_{pj}$$

即 $C_{ij}$ 是 $\mathbf{A}$ 第 $i$ 行 · $\mathbf{B}$ 第 $j$ 列的**内积**。
i.e. $C_{ij}$ is the inner product of row $i$ of $\mathbf{A}$ and column $j$ of $\mathbf{B}$.

### 视角 2：列视角 / Column view

$\mathbf{C}$ 的**第 $j$ 列** = $\mathbf{A}$ 作用在 $\mathbf{B}$ 的第 $j$ 列上：
**Column $j$ of $\mathbf{C}$** = $\mathbf{A}$ applied to column $j$ of $\mathbf{B}$:

$$\mathbf{C}_{:,j} = \mathbf{A} \cdot \mathbf{B}_{:,j}$$

### 视角 3：外积和 / Sum of outer products

$$\mathbf{C} = \sum_{p=1}^{k} \mathbf{A}_{:,p}\,\mathbf{B}_{p,:}$$

即每一对（$\mathbf{A}$ 的第 $p$ 列）× （$\mathbf{B}$ 的第 $p$ 行）= 一个 $m \times n$ 矩阵；把所有 $k$ 个加起来。
Each pair (col $p$ of $\mathbf{A}$) × (row $p$ of $\mathbf{B}$) = an $m \times n$ matrix; sum all $k$ pairs.

> ⭐ **视角 3 是 SVD 的灵魂**：SVD 把 $\mathbf{A}$ 拆成一堆"秩 1 矩阵"的加和。
> View 3 is the soul of SVD: it decomposes $\mathbf{A}$ into a sum of rank-1 matrices.


In [ ]:
# 三种视角实现矩阵乘，验证等价 / Three implementations
rng = np.random.default_rng(0)
A = rng.normal(size=(3, 4))
B = rng.normal(size=(4, 2))

# 视角 1：标准定义 / Standard
m, k = A.shape; _, n = B.shape
C1 = np.zeros((m, n))
for i in range(m):
    for j in range(n):
        for p in range(k):
            C1[i, j] += A[i, p] * B[p, j]

# 视角 2：列视角 / Column view
C2 = np.zeros((m, n))
for j in range(n):
    C2[:, j] = A @ B[:, j]

# 视角 3：外积和 / Sum of outer products
C3 = np.zeros((m, n))
for p in range(k):
    C3 += np.outer(A[:, p], B[p, :])

C_numpy = A @ B
print("all equal?", np.allclose(C1, C_numpy) and
      np.allclose(C2, C_numpy) and np.allclose(C3, C_numpy))


### 矩阵乘法的性质 / Properties

- **结合律 / Associative**：$(\mathbf{A}\mathbf{B})\mathbf{C} = \mathbf{A}(\mathbf{B}\mathbf{C})$
- **分配律 / Distributive**：$\mathbf{A}(\mathbf{B}+\mathbf{C}) = \mathbf{A}\mathbf{B} + \mathbf{A}\mathbf{C}$
- **❗不满足交换律 / NOT commutative**：一般 $\mathbf{A}\mathbf{B} \ne \mathbf{B}\mathbf{A}$
- **转置反序 / Transpose reverses**：$(\mathbf{A}\mathbf{B})^\top = \mathbf{B}^\top \mathbf{A}^\top$


<a id="7"></a>
## 7. 特殊矩阵 / Special Matrices

| 类型 / Type | 定义 / Definition | 性质 / Property |
|---|---|---|
| 方阵 / Square | $m = n$ | 可以谈逆、特征值 / can have inverse, eigenvalues |
| 对角 / Diagonal | $A_{ij} = 0$ for $i\ne j$ | 计算超快 / fast ops |
| 单位阵 / Identity $\mathbf{I}$ | 对角线全 1 | $\mathbf{A}\mathbf{I} = \mathbf{A}$ |
| 对称 / Symmetric | $\mathbf{A}^\top = \mathbf{A}$ | 协方差矩阵就是这种 / cov matrices |
| 正交 / Orthogonal | $\mathbf{Q}^\top \mathbf{Q} = \mathbf{I}$ | 列向量两两正交且单位长 |
| 正定 / Positive definite | $\mathbf{x}^\top \mathbf{A} \mathbf{x} > 0,\;\forall \mathbf{x}\ne\mathbf{0}$ | 凸优化的核心 / core of convex opt |

### 正交矩阵的"超能力" / The superpower of orthogonal matrices

$\mathbf{Q}^\top \mathbf{Q} = \mathbf{I}$ 意味着 $\mathbf{Q}^{-1} = \mathbf{Q}^\top$ —— **逆 = 转置**，超快！
$\mathbf{Q}^{-1} = \mathbf{Q}^\top$ — inverse = transpose, blazing fast.

几何上：正交矩阵 = **保长度、保夹角**的变换（旋转 + 反射）。
Geometric: orthogonal = **rotation + reflection** (preserves lengths and angles).


In [ ]:
# 演示正交矩阵 / Orthogonal matrix demo
theta = np.pi / 6
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

print("Q =\n", Q)
print("\nQᵀ Q =\n", Q.T @ Q)         # 应当是 I / should be identity
print("\nQ⁻¹ =\n", np.linalg.inv(Q))
print("Qᵀ =\n", Q.T)
print("equal?", np.allclose(Q.T, np.linalg.inv(Q)))

# 保长度：‖Q v‖ = ‖v‖
v = np.array([3.0, 4.0])
print(f"\n‖v‖  = {np.linalg.norm(v):.4f}")
print(f"‖Qv‖ = {np.linalg.norm(Q @ v):.4f}")


<a id="8"></a>
## 8. 行列式 / Determinant —— 几何：面积/体积缩放

$\det(\mathbf{A})$ 是一个标量，**它的绝对值 = 变换后 $\mathbf{A}$ 把单位正方形/立方体的面积/体积放大了多少倍**。
$\lvert\det(\mathbf{A})\rvert$ = how much $\mathbf{A}$ scales unit area / volume.

- $\det(\mathbf{A}) > 0$ → 保持方向 / preserves orientation
- $\det(\mathbf{A}) < 0$ → 翻转方向（含反射）/ flips orientation (reflection)
- $\det(\mathbf{A}) = 0$ → **变换把空间压扁了** / collapses dimension（不可逆）

对 $2\times 2$ 矩阵：
$$\det \begin{pmatrix} a & b \\\\ c & d \end{pmatrix} = ad - bc$$


In [ ]:
# 不同 |det| 的可视化 / Visualize different determinants
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, (name, A) in zip(axes, [
    ("identity\ndet=1",       np.eye(2)),
    ("stretch×2.5\ndet=2.5",  np.array([[2.5, 0], [0, 1]])),
    ("reflect\ndet=-1",       np.array([[1, 0], [0, -1]])),
    ("singular\ndet=0",       np.array([[1, 2], [2, 4]])),
]):
    out = A @ square
    ax.fill(square[0], square[1], color="gray", alpha=0.3)
    ax.fill(out[0], out[1], color="steelblue", alpha=0.5)
    ax.plot(out[0], out[1], "b-")
    ax.set_xlim(-2, 4); ax.set_ylim(-2, 4)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(f"{name}\nnumpy det = {np.linalg.det(A):.2f}")

plt.tight_layout()
plt.show()


**最后一张 singular 例子**：$\det = 0$ 时变换把整个 2D 平面**压扁成一条直线**——丢了一个维度，**不可逆**。
The singular case collapses 2D to a line — loses a dimension, **not invertible**.

### 行列式的代数性质 / Algebraic properties

| 性质 / Property | 公式 / Formula |
|---|---|
| 乘积 / Product | $\det(\mathbf{A}\mathbf{B}) = \det(\mathbf{A})\det(\mathbf{B})$ |
| 转置 / Transpose | $\det(\mathbf{A}^\top) = \det(\mathbf{A})$ |
| 逆 / Inverse | $\det(\mathbf{A}^{-1}) = 1/\det(\mathbf{A})$ |
| 行交换 / Row swap | 变号 / flips sign |


<a id="9"></a>
## 9. 秩、列空间、零空间 / Rank, Column Space, Null Space

### 9.1 列空间 / Column space

$$\mathcal{C}(\mathbf{A}) = \mathrm{span}\{\mathbf{A}_{:,1}, \dots, \mathbf{A}_{:,n}\}$$

= 矩阵所有列张成的子空间 = $\mathbf{A}\mathbf{x}$ 能取到的**所有输出**的集合。
The subspace spanned by columns = all reachable outputs $\mathbf{A}\mathbf{x}$.

### 9.2 秩 / Rank

$$\mathrm{rank}(\mathbf{A}) = \dim \mathcal{C}(\mathbf{A})$$

= 列空间的维度 = **线性独立列的数目**。
= dimension of column space = number of linearly independent columns.

性质：$\mathrm{rank}(\mathbf{A}) \le \min(m, n)$，等号成立时叫**满秩 / full rank**。
Property: $\mathrm{rank}(\mathbf{A}) \le \min(m, n)$; equality = **full rank**.

### 9.3 零空间 / Null space

$$\mathcal{N}(\mathbf{A}) = \{\mathbf{x} : \mathbf{A}\mathbf{x} = \mathbf{0}\}$$

= 被"压扁到原点"的所有方向。
= all directions that get squashed to the origin.

**秩-零度定理 / Rank-Nullity**：$\mathrm{rank}(\mathbf{A}) + \dim \mathcal{N}(\mathbf{A}) = n$。


In [ ]:
# 求秩 / Compute rank
A1 = np.array([[1, 2, 3],
               [4, 5, 6],
               [7, 8, 9]])
print(f"A1 rank: {np.linalg.matrix_rank(A1)}")
# 这个矩阵的第 3 行 = 2 * 第 2 行 - 第 1 行，所以秩只有 2

A2 = np.array([[1, 0, 0],
               [0, 1, 0],
               [0, 0, 1]])
print(f"A2 rank: {np.linalg.matrix_rank(A2)}")     # 3，满秩 / full

# rank-1 矩阵：两列共线 / Rank-1
A3 = np.outer([1, 2, 3], [4, 5])      # 3×2 矩阵，列空间维度只有 1
print(f"A3 =\n{A3}")
print(f"A3 rank: {np.linalg.matrix_rank(A3)}")


<a id="10"></a>
## 10. 矩阵的逆 / Matrix Inverse

$\mathbf{A}^{-1}$ 满足：
$$\mathbf{A}\mathbf{A}^{-1} = \mathbf{A}^{-1}\mathbf{A} = \mathbf{I}$$

**只有方阵 + 满秩**的矩阵才可逆。
**Only square + full-rank** matrices are invertible.

### 几个等价说法 / Equivalent statements

$\mathbf{A} \in \mathbb{R}^{n\times n}$ 可逆 $\iff$:
- $\det(\mathbf{A}) \ne 0$
- $\mathrm{rank}(\mathbf{A}) = n$
- 列向量线性独立 / columns linearly independent
- $\mathcal{N}(\mathbf{A}) = \{\mathbf{0}\}$
- $\mathbf{A}\mathbf{x} = \mathbf{b}$ 对**任何** $\mathbf{b}$ 都有唯一解 / unique solution for any $\mathbf{b}$

### 实战 / In practice：**别真的算逆**！

```python
# ❌ 慢 + 不稳定 / slow + unstable
x = np.linalg.inv(A) @ b

# ✅ 快 + 稳定 / fast + stable
x = np.linalg.solve(A, b)
```


In [ ]:
# 演示逆 / Inverse demo
A = np.array([[4, 7], [2, 6]])
A_inv = np.linalg.inv(A)
print("A =\n", A)
print("\nA⁻¹ =\n", A_inv)
print("\nA · A⁻¹ =\n", A @ A_inv)            # 接近单位阵 / near identity
print(f"\ndet(A) = {np.linalg.det(A):.4f}")


<a id="11"></a>
## 11. 解线性方程组 / Solving Linear Systems $\mathbf{A}\mathbf{x} = \mathbf{b}$

三种情况 / Three cases:

| 情况 / Case | 几何 / Geometric | 求解 / Solver |
|---|---|---|
| $m = n$，$\mathbf{A}$ 满秩 / square full-rank | 唯一交点 / unique intersection | `solve(A, b)` |
| $m > n$（超定 / over-det.） | 一般无解 → **最小二乘** / no exact solution → LSQ | `lstsq(A, b)` |
| $m < n$（欠定 / under-det.） | 无穷多解 / infinite solutions | `lstsq(A, b)` (gives min-norm) |


In [ ]:
# Case 1: 方阵满秩 / Square full-rank — unique solution
A = np.array([[2., 1.], [1., 3.]])
b = np.array([5., 10.])

x = np.linalg.solve(A, b)
print(f"x = {x}")
print(f"check Ax: {A @ x}")    # should equal b


In [ ]:
# Case 2: 超定（行 > 列）→ 最小二乘 / Overdetermined → least squares
# 这就是线性回归的核心！/ This IS linear regression.
#
# 目标 / Goal:  min ‖A x - b‖²
# 解 / Solution: x = (AᵀA)⁻¹ Aᵀb   (正规方程)
#
np.random.seed(0)
m, n = 50, 3
A = np.random.randn(m, n)
true_x = np.array([1.5, -2.0, 0.5])
b = A @ true_x + 0.1 * np.random.randn(m)

x_hat, residuals, rank, sv = np.linalg.lstsq(A, b, rcond=None)
print(f"true x  : {true_x}")
print(f"x_hat   : {x_hat.round(3)}")
print(f"residual L²: {residuals[0]:.4f}")


<a id="12"></a>
## 12. 投影 / Projection

### 12.1 把向量 $\mathbf{b}$ 投影到向量 $\mathbf{a}$

$$\mathrm{proj}_{\mathbf{a}}\mathbf{b} = \dfrac{\langle \mathbf{a}, \mathbf{b}\rangle}{\|\mathbf{a}\|^2}\,\mathbf{a}$$

证明思路 / Derivation: 找 $c$ 使 $\mathbf{b} - c\mathbf{a} \perp \mathbf{a}$ → $\mathbf{a}^\top(\mathbf{b} - c\mathbf{a}) = 0$ → $c = \mathbf{a}^\top\mathbf{b} / \|\mathbf{a}\|^2$。
Idea: pick $c$ so the residual is orthogonal to $\mathbf{a}$.

### 12.2 投影到子空间（列空间）/ Project onto a subspace

设 $\mathbf{A}$ 列线性独立，要把 $\mathbf{b}$ 投影到 $\mathcal{C}(\mathbf{A})$。
With independent columns, project $\mathbf{b}$ onto $\mathcal{C}(\mathbf{A})$:

$$\boxed{\;\hat{\mathbf{b}} = \mathbf{A}(\mathbf{A}^\top\mathbf{A})^{-1}\mathbf{A}^\top \mathbf{b}\;}$$

这就是最小二乘的几何解释——"把 $\mathbf{b}$ 投影到 $\mathbf{A}$ 列空间得到的就是最近的可达点"。
This **is** the geometric meaning of least squares — project $\mathbf{b}$ onto $\mathbf{A}$'s column space.


In [ ]:
# 把 b 投影到 a / Project b onto a
a = np.array([3.0, 1.0])
b = np.array([1.0, 4.0])

proj = (a @ b) / (a @ a) * a
print(f"proj_a(b) = {proj}")
print(f"residual  = {b - proj} (should be ⟂ a)")
print(f"check ⟂   : (b - proj) · a = {(b - proj) @ a:.6f}")  # ~0

# 可视化 / Viz
fig, ax = plt.subplots(figsize=(6, 6))
ax.quiver(0, 0, a[0], a[1], angles="xy", scale_units="xy", scale=1, color="blue", width=0.012, label="a")
ax.quiver(0, 0, b[0], b[1], angles="xy", scale_units="xy", scale=1, color="green", width=0.012, label="b")
ax.quiver(0, 0, proj[0], proj[1], angles="xy", scale_units="xy", scale=1, color="red", width=0.012, label="proj")
ax.plot([b[0], proj[0]], [b[1], proj[1]], "k--", label="residual")
ax.set_xlim(-1, 5); ax.set_ylim(-1, 5); ax.set_aspect("equal")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.grid(alpha=0.3); ax.legend()
ax.set_title("Project b onto a")
plt.show()


<a id="13"></a>
## 13. 特征值与特征向量 ⭐ / Eigenvalues & Eigenvectors

### 13.1 定义 / Definition

非零向量 $\mathbf{v}$ 满足
$$\mathbf{A}\mathbf{v} = \lambda \mathbf{v}, \quad \lambda \in \mathbb{R}$$
则 $\lambda$ 叫**特征值** / eigenvalue，$\mathbf{v}$ 叫**特征向量** / eigenvector。

**几何意义** / Geometric meaning: $\mathbf{v}$ 是 $\mathbf{A}$ 作用下**不旋转**只**拉伸**的方向，拉伸因子就是 $\lambda$。
$\mathbf{v}$ is a direction $\mathbf{A}$ **only stretches without rotating**; the stretch factor is $\lambda$.

### 13.2 怎么求 / How to find

特征值满足**特征方程** / Characteristic equation:
$$\det(\mathbf{A} - \lambda \mathbf{I}) = 0$$

对每个 $\lambda$，零空间 $\mathcal{N}(\mathbf{A} - \lambda \mathbf{I})$ 给出对应的特征向量。
For each $\lambda$, the null space of $(\mathbf{A} - \lambda\mathbf{I})$ gives the eigenvectors.


In [ ]:
# 演示：对一个对称矩阵求特征值/向量 / Eigen of a symmetric matrix
A = np.array([[3., 1.],
              [1., 3.]])

eigvals, eigvecs = np.linalg.eig(A)
print("eigenvalues  :", eigvals)            # [4, 2]
print("eigenvectors :\n", eigvecs)          # 每一列是一个特征向量

# 验证 A v = λ v / Verify
for i in range(2):
    v = eigvecs[:, i]
    lam = eigvals[i]
    print(f"\nA v_{i} = {A @ v}")
    print(f"λ_{i} v_{i} = {lam * v}")


In [ ]:
# 可视化：特征向量在变换下不旋转 / Eigenvectors don't rotate
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# 左：原始向量 / Left: original
ax = axes[0]
ax.quiver(0, 0, eigvecs[0, 0], eigvecs[1, 0], color="red", angles="xy",
          scale_units="xy", scale=1, width=0.015, label=f"v₁ (λ={eigvals[0]:.0f})")
ax.quiver(0, 0, eigvecs[0, 1], eigvecs[1, 1], color="blue", angles="xy",
          scale_units="xy", scale=1, width=0.015, label=f"v₂ (λ={eigvals[1]:.0f})")
# 一个非特征向量 / a non-eigen vector
u = np.array([1.0, 0.5])
u = u / np.linalg.norm(u)
ax.quiver(0, 0, u[0], u[1], color="green", angles="xy",
          scale_units="xy", scale=1, width=0.015, label="u (non-eigen)")
ax.set_xlim(-2, 5); ax.set_ylim(-2, 5); ax.set_aspect("equal")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.grid(alpha=0.3); ax.legend(); ax.set_title("Before transformation")

# 右：A 作用后 / Right: after A
ax = axes[1]
for vec, col, lab in zip([eigvecs[:, 0], eigvecs[:, 1], u],
                          ["red", "blue", "green"],
                          ["A v₁ (still aligned!)", "A v₂ (still aligned!)", "A u (ROTATED)"]):
    out = A @ vec
    ax.quiver(0, 0, out[0], out[1], color=col, angles="xy",
              scale_units="xy", scale=1, width=0.015, label=lab)
ax.set_xlim(-2, 5); ax.set_ylim(-2, 5); ax.set_aspect("equal")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.grid(alpha=0.3); ax.legend(); ax.set_title("After A")

plt.tight_layout(); plt.show()


**看右图**：红蓝箭头（特征向量）**还在原来的方向上**，只是长度变了；绿色箭头**转向**了。
**Look at the right plot**: the red/blue arrows (eigenvectors) **stay on the same line**, only stretched; the green one **rotates**.

### 13.3 特征分解 / Eigendecomposition

如果 $\mathbf{A} \in \mathbb{R}^{n\times n}$ 有 $n$ 个线性独立特征向量，可写成：
If $\mathbf{A}$ has $n$ independent eigenvectors:

$$\boxed{\;\mathbf{A} = \mathbf{V} \boldsymbol{\Lambda} \mathbf{V}^{-1}\;}$$

其中 $\mathbf{V}$ 列是特征向量，$\boldsymbol{\Lambda}$ 是对角阵装特征值。
where columns of $\mathbf{V}$ are eigenvectors and $\boldsymbol{\Lambda}$ is diagonal with eigenvalues.

**用途 / Use cases**:
- $\mathbf{A}^k = \mathbf{V}\boldsymbol{\Lambda}^k \mathbf{V}^{-1}$ —— 矩阵高次方一秒算出
- 分析动力系统、Markov chain 收敛
- PCA, spectral clustering, PageRank


<a id="14"></a>
## 14. 对称矩阵的谱分解 / Spectral Theorem

### 定理 / Theorem

设 $\mathbf{A} \in \mathbb{R}^{n\times n}$ 为**对称**矩阵（$\mathbf{A}^\top = \mathbf{A}$），则：
For a symmetric matrix $\mathbf{A}$:

1. 所有特征值都是**实数** / all eigenvalues are real
2. 可以选出**两两正交**的特征向量 / can choose pairwise orthogonal eigenvectors
3. 可写成 $\mathbf{A} = \mathbf{Q}\boldsymbol{\Lambda}\mathbf{Q}^\top$，其中 $\mathbf{Q}$ **正交** / where $\mathbf{Q}$ is orthogonal

$$\boxed{\;\mathbf{A} = \mathbf{Q}\,\boldsymbol{\Lambda}\,\mathbf{Q}^\top, \quad \mathbf{Q}^\top \mathbf{Q} = \mathbf{I}\;}$$

> 💡 **为什么 DS 特别关心这个 / Why this matters for DS**
> 协方差矩阵就是对称的！PCA 本质上就是协方差矩阵的谱分解。
> Covariance matrices ARE symmetric. PCA is literally the spectral decomposition of the covariance matrix.

### 实数特征值的简证 / Sketch of "real eigenvalues"

设 $\mathbf{A}\mathbf{v} = \lambda \mathbf{v}$, $\mathbf{v} \in \mathbb{C}^n$。两边乘 $\bar{\mathbf{v}}^\top$:

$$\bar{\mathbf{v}}^\top \mathbf{A} \mathbf{v} = \lambda \bar{\mathbf{v}}^\top \mathbf{v}$$

取共轭转置 + $\mathbf{A}^\top = \mathbf{A}$ (实对称):

$$\overline{(\bar{\mathbf{v}}^\top \mathbf{A} \mathbf{v})} = \bar{\lambda}\,\overline{\bar{\mathbf{v}}^\top \mathbf{v}}$$

左边等于自身（标量取共轭），右边 $\bar{\mathbf{v}}^\top \mathbf{v} > 0$，所以 $\lambda = \bar{\lambda}$，即 **$\lambda$ 实数**。
LHS equals itself; since $\bar{\mathbf{v}}^\top \mathbf{v} > 0$, we get $\lambda = \bar{\lambda}$, i.e. $\lambda$ is real.


In [ ]:
# 对称矩阵：用 eigh 而不是 eig / Use eigh for symmetric matrices
S = np.array([[4., 1., 2.],
              [1., 3., 0.],
              [2., 0., 5.]])
print(f"is symmetric? {np.allclose(S, S.T)}")

# eigh 专门给对称/Hermitian 矩阵用 —— 更快、保证实数返回
# eigh is faster and guarantees real outputs for symmetric / Hermitian
eigvals, Q = np.linalg.eigh(S)
print(f"\neigenvalues: {eigvals}")
print(f"\nQ (columns are eigenvectors):\n{Q}")

# 验证正交 / Verify orthogonality
print(f"\nQᵀQ =\n{(Q.T @ Q).round(8)}")    # 应为单位阵 / identity

# 重构 / Reconstruct
Lambda = np.diag(eigvals)
S_recon = Q @ Lambda @ Q.T
print(f"\nQ Λ Qᵀ matches S? {np.allclose(S_recon, S)}")


<a id="15"></a>
## 15. SVD ⭐ / Singular Value Decomposition

**线性代数的终极武器**。
**The ultimate weapon of linear algebra.**

### 定理 / Theorem

任何矩阵 $\mathbf{A} \in \mathbb{R}^{m\times n}$ —— **不要求方阵，不要求满秩** —— 都可以分解成：
Any matrix (no squareness or full-rank requirement) can be decomposed as:

$$\boxed{\;\mathbf{A} \;=\; \mathbf{U}\,\boldsymbol{\Sigma}\,\mathbf{V}^\top\;}$$

其中：
- $\mathbf{U} \in \mathbb{R}^{m\times m}$ 正交 —— 左奇异向量 / left singular vectors
- $\boldsymbol{\Sigma} \in \mathbb{R}^{m\times n}$ 对角，对角线上是非负**奇异值** $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$
- $\mathbf{V} \in \mathbb{R}^{n\times n}$ 正交 —— 右奇异向量 / right singular vectors

### 几何意义 / Geometric meaning

$$
\mathbf{A}\mathbf{x} \;=\; \underbrace{\mathbf{U}}_{\text{旋转}} \;\underbrace{\boldsymbol{\Sigma}}_{\text{沿坐标轴拉伸}} \;\underbrace{\mathbf{V}^\top}_{\text{旋转}} \;\mathbf{x}
$$

**任何线性变换 = 一次旋转 + 沿坐标轴拉伸 + 再一次旋转。**
**Every linear transformation = rotate → axis-aligned stretch → rotate.**

### 数值上 / Numerically

奇异值告诉你 $\mathbf{A}$ 各方向"有多重要"。**截掉小的 $\sigma_k$ 就得到最优低秩近似**——这是图像压缩、推荐系统、LSA 的原理。
Singular values rank "directions of importance". **Dropping small $\sigma_k$ gives the optimal low-rank approximation** — foundation of image compression, recommenders, LSA.


In [ ]:
# SVD 演示 / SVD demo
rng = np.random.default_rng(0)
A = rng.normal(size=(4, 3))

U, s, Vt = np.linalg.svd(A, full_matrices=False)
print(f"A.shape = {A.shape}")
print(f"U.shape = {U.shape}")
print(f"s       = {s.round(3)}    (singular values, descending)")
print(f"Vt.shape= {Vt.shape}")

# 重构 / Reconstruct
A_recon = U @ np.diag(s) @ Vt
print(f"\nreconstruction matches A? {np.allclose(A_recon, A)}")


### Eckart–Young 定理 / Eckart–Young theorem

**对秩-$k$ 近似**，$\mathbf{A}_k = \sum_{i=1}^k \sigma_i \mathbf{u}_i \mathbf{v}_i^\top$ 是**最优近似**：

$$\mathbf{A}_k = \arg\min_{\mathrm{rank}(\mathbf{B}) \le k} \|\mathbf{A} - \mathbf{B}\|_F$$

并且误差精确等于 $\|\mathbf{A} - \mathbf{A}_k\|_F = \sqrt{\sigma_{k+1}^2 + \dots + \sigma_r^2}$。
And the error equals the L2 norm of the discarded singular values.

⭐ 这就是 **PCA = SVD on centered data** 的根本原因。
This is **why PCA = SVD on centered data**.


In [ ]:
# 用 SVD 做"图像压缩"演示 / Image compression via SVD
# 造一张人造灰度图 / Make a synthetic grayscale image
xx, yy = np.meshgrid(np.linspace(-3, 3, 80), np.linspace(-3, 3, 80))
img = np.exp(-(xx**2 + yy**2)) + 0.5 * np.exp(-((xx-1.5)**2 + (yy-1.5)**2))

U, s, Vt = np.linalg.svd(img, full_matrices=False)

ks = [1, 5, 20, 80]
fig, axes = plt.subplots(1, len(ks), figsize=(13, 3.5))
for ax, k in zip(axes, ks):
    img_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    energy = s[:k].sum() / s.sum() * 100
    ax.imshow(img_k, cmap="gray")
    ax.set_title(f"rank = {k}\n{energy:.1f}% energy")
    ax.axis("off")
plt.suptitle("Low-rank SVD approximations")
plt.tight_layout()
plt.show()


**观察 / Observation**：
- $k=1$：丢了细节但主体形状已经出来了 / lost detail but main shape visible
- $k=5$：肉眼几乎和原图一样 / almost indistinguishable
- 80 个奇异值实际只需要 5 个就够 → **数据有强低秩结构**
- This data has strong **low-rank structure** — only ~5 components are needed.

这就是为什么 JPEG / 推荐系统 / LLM 的 LoRA 都能"用 1% 的参数表达 99% 的信息"。
This is why JPEG / recommenders / LoRA can compress to ~1% of params while keeping ~99% of info.


<a id="16"></a>
## 16. 应用：在 Iris 上手写 PCA / PCA from Scratch on Iris

PCA 把所有上面这些一锅烩了：协方差矩阵 + 特征分解（或 SVD）+ 投影。
PCA combines: covariance + eigendecomposition (or SVD) + projection.

### 数学推导 / Derivation

设数据 $\mathbf{X} \in \mathbb{R}^{n\times d}$ 已**中心化**（每列均值为 0）。

**目标**：找单位方向 $\mathbf{u} \in \mathbb{R}^d$ 使投影方差最大。
**Goal**: find unit direction $\mathbf{u}$ maximizing projection variance.

投影到 $\mathbf{u}$ 上的数据是 $\mathbf{X}\mathbf{u} \in \mathbb{R}^n$，方差是：
Projected data variance:

$$\mathrm{Var}(\mathbf{X}\mathbf{u}) = \frac{1}{n}(\mathbf{X}\mathbf{u})^\top(\mathbf{X}\mathbf{u}) = \mathbf{u}^\top \underbrace{\frac{1}{n}\mathbf{X}^\top\mathbf{X}}_{\boldsymbol{\Sigma}} \mathbf{u} = \mathbf{u}^\top \boldsymbol{\Sigma}\mathbf{u}$$

**优化** / Constrained optimization (Lagrangian):

$$\max_{\mathbf{u}} \mathbf{u}^\top \boldsymbol{\Sigma} \mathbf{u} \quad \text{s.t.} \quad \|\mathbf{u}\| = 1$$

拉格朗日法 ⇒ $\boldsymbol{\Sigma}\mathbf{u} = \lambda \mathbf{u}$。

**结论 / Conclusion**:
- 最大特征值 $\lambda_1$ 对应的特征向量 $\mathbf{u}_1$ = **第一主成分** / first principal component
- 第二主成分是次大特征值对应的特征向量，以此类推

> 💡 PCA = 求中心化数据**协方差矩阵的谱分解**。这就是为什么协方差矩阵必须对称（已经满足！）。
> PCA = spectral decomposition of the covariance matrix of centered data.


In [ ]:
# 加载 Iris 并中心化 / Load Iris and center
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target
print(f"X.shape = {X.shape}     (n samples × d features)")

# Step 1: 中心化 / Center
X_c = X - X.mean(axis=0, keepdims=True)
print(f"means after centering: {X_c.mean(axis=0).round(8)}")    # ~0


In [ ]:
# Step 2: 协方差矩阵 / Covariance matrix
n = X_c.shape[0]
Sigma = (X_c.T @ X_c) / (n - 1)        # 注意 n-1 是无偏 / n-1 for unbiased
print("Σ =\n", Sigma.round(3))
print(f"\nis symmetric? {np.allclose(Sigma, Sigma.T)}")


In [ ]:
# Step 3: 谱分解（对称矩阵用 eigh）/ Spectral decomp via eigh
eigvals, eigvecs = np.linalg.eigh(Sigma)

# eigh 默认升序，反转成降序 / eigh returns ascending — flip
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print(f"eigenvalues (= variances along PCs): {eigvals.round(3)}")
print(f"\n%explained: {(eigvals / eigvals.sum() * 100).round(1)}")
print(f"\ncumulative: {(eigvals.cumsum() / eigvals.sum() * 100).round(1)} %")


**前 2 个主成分已经解释 97.8% 的方差** —— Iris 在 4 维空间里本质上是 **2 维的**。
**The first 2 PCs explain 97.8% of variance** — Iris is essentially 2-D in 4-D space.


In [ ]:
# Step 4: 投影到前 2 个主成分 / Project onto first 2 PCs
X_pca = X_c @ eigvecs[:, :2]
print(f"X_pca.shape = {X_pca.shape}")

# 可视化 / Visualize
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# 左：原始 4D 中任意挑 2 维 / Left: 2 raw dims
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=40, edgecolor="k")
axes[0].set_xlabel(iris.feature_names[0])
axes[0].set_ylabel(iris.feature_names[1])
axes[0].set_title("Raw features (sepal length × width)")

# 右：PCA 投影 / Right: PCA projection
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="viridis", s=40, edgecolor="k")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("PCA projection (97.8% variance)")

plt.tight_layout()
plt.show()


In [ ]:
# Step 5: 和 sklearn 对照 / Cross-check with sklearn
from sklearn.decomposition import PCA

ref = PCA(n_components=2).fit(X)
X_pca_sk = ref.transform(X)

print(f"sklearn explained variance ratio: {ref.explained_variance_ratio_.round(3)}")
print(f"ours                            : {(eigvals[:2] / eigvals.sum()).round(3)}")

# 主成分方向可能差一个符号（特征向量 ±v 都对），所以取 abs
print(f"\nmax |diff of |X_pca||: {np.abs(np.abs(X_pca) - np.abs(X_pca_sk)).max():.2e}")


**结果完全一致**——我们仅用 NumPy 复现了 sklearn 的 PCA。
**Identical results** — we replicated sklearn's PCA with bare NumPy.

📝 这就把本节所有核心概念串了起来：
This pulled together everything in this lesson:
- 矩阵乘法 / matrix multiplication（$\mathbf{X}^\top\mathbf{X}$）
- 对称矩阵 / symmetric matrices（协方差矩阵）
- 谱分解 / spectral decomposition
- 投影 / projection（投到主成分张成的子空间）
- 特征值含义 / eigenvalue interpretation（沿该方向的方差）


<a id="17"></a>
## 17. 小结 / Summary

### 核心概念地图 / Concept map

```
向量 (vector)  ─────────┐
   │                    │
   ├─ 内积 ⟨v,w⟩         ├─ 范数 / 距离 / 余弦
   │                    │
矩阵 (matrix)            │
   │                    │
   ├─ 线性变换 ⭐         │      ┌── 特征值/向量 ⭐
   ├─ 矩阵乘法 (3 视角)   │     │       │
   ├─ 行列式 = 体积缩放   │     │       └─ 谱定理 (对称 → 正交)
   ├─ 秩 / 列空间         │     │            │
   ├─ 投影                │     │            └─ PCA
   │                    │     │
   └─────── 解 Ax=b ←─────┘     └── SVD ⭐ (无方阵要求)
                                       │
                                       ├─ 图像压缩
                                       ├─ 推荐系统
                                       ├─ LoRA / LSA
                                       └─ Eckart–Young 最优低秩
```

### 💡 工业速查 / Industry cheat sheet

| 任务 / Task | 数学 | NumPy |
|---|---|---|
| 求范数 | $\|\mathbf{x}\|_p$ | `np.linalg.norm(x, ord=p)` |
| 余弦相似度 | $\mathbf{x}^\top\mathbf{y}/(\|\mathbf{x}\|\|\mathbf{y}\|)$ | `x @ y / (norm(x)*norm(y))` |
| 解 $\mathbf{A}\mathbf{x} = \mathbf{b}$ | — | `np.linalg.solve(A, b)` |
| 最小二乘 | $\hat{\mathbf{x}} = (\mathbf{A}^\top\mathbf{A})^{-1}\mathbf{A}^\top\mathbf{b}$ | `np.linalg.lstsq(A, b, rcond=None)` |
| 求秩 | $\dim \mathcal{C}(\mathbf{A})$ | `np.linalg.matrix_rank(A)` |
| 行列式 | $\det \mathbf{A}$ | `np.linalg.det(A)` |
| 对称矩阵特征 | $\mathbf{A} = \mathbf{Q}\boldsymbol{\Lambda}\mathbf{Q}^\top$ | `np.linalg.eigh(A)` |
| 一般矩阵特征 | $\mathbf{A} = \mathbf{V}\boldsymbol{\Lambda}\mathbf{V}^{-1}$ | `np.linalg.eig(A)` |
| SVD | $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$ | `np.linalg.svd(A, full_matrices=False)` |
| 投影矩阵 | $\mathbf{P} = \mathbf{A}(\mathbf{A}^\top\mathbf{A})^{-1}\mathbf{A}^\top$ | 直接表达 |

### 💡 面试必背 / Interview must-knows

1. **$\mathbf{A}\mathbf{x}$ 三视角**：标量点积 / 列组合 / 列空间元素
2. **特征值几何**：$\mathbf{A}$ 不旋转只拉伸的方向
3. **对称矩阵**：实特征值 + 正交特征向量（PCA 的基础）
4. **SVD = 旋转-拉伸-旋转**
5. **正规方程 + Eckart–Young** = 线性回归 / PCA 的最优性证明
6. **正交矩阵**：$\mathbf{Q}^{-1} = \mathbf{Q}^\top$，保长度保角度

### 下一节预告 / Next up

**Part 0.8 · 微积分** —— 梯度、链式法则、雅各比、海森——为反向传播和优化算法打底。
**Part 0.8 · Calculus** — gradients, chain rule, Jacobian, Hessian — prereqs for backprop and optimization.
